In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Demand_Forecasting_and_Inventory_Optimization\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Demand_Forecasting_and_Inventory_Optimization'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from src.demand_forecasting_and_inventory_optimization.constants import *
from src.demand_forecasting_and_inventory_optimization.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH, schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [9]:
import os
import zipfile
from src.demand_forecasting_and_inventory_optimization import logger
from src.demand_forecasting_and_inventory_optimization.utils.common import get_size
import requests


In [10]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''

        try: 
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            response = requests.get(dataset_url, stream=True, timeout=60)

            response.raise_for_status()

            with open(zip_download_dir, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            print("Download completed")

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e
        
    

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Extracted zip file {self.config.local_data_file} into dir {unzip_path}")



In [11]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-07-01 12:23:37,604: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-01 12:23:37,607: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-01 12:23:37,630: INFO: common: yaml file: config\schema.yaml loaded successfully]
[2026-07-01 12:23:37,632: INFO: common: created directory at: artifacts]
[2026-07-01 12:23:37,634: INFO: common: created directory at: artifacts/data_ingestion]
[2026-07-01 12:23:37,635: INFO: 859167278: Downloading data from https://github.com/ajaychaudhary8104/Datasets/raw/refs/heads/main/data.zip into file artifacts/data_ingestion/data.zip]
Download completed
[2026-07-01 12:23:42,082: INFO: 859167278: Downloaded data from https://github.com/ajaychaudhary8104/Datasets/raw/refs/heads/main/data.zip into file artifacts/data_ingestion/data.zip]
[2026-07-01 12:23:42,194: INFO: 859167278: Extracted zip file artifacts/data_ingestion/data.zip into dir artifacts/data_ingestion]
